# LaMa: Large Mask Inpainting for Mughal Heritage

## V2: 512x512 High Resolution

**Team Labaik**
- Talha Abid (L1F23BSAI0058)
- Abdullah Zafar (L1F23BSAI0054)
- Abdul Rehman (L1F23BSAI0050)
- Waqar Hussain (L1F23BSAI0078)

**Configuration:**
- Resolution: 512×512
- Batch Size: 16
- Epochs: 100
- Target: Beat OpenCV (21.27 dB)

In [ ]:
# Setup and GPU Check
import os
import sys
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import random
import json
from pathlib import Path

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

print(f"\nNumPy version: {np.__version__}")
print(f"OpenCV version: {cv2.__version__}")

In [ ]:
# PyTorch and Device Setup with Mixed Precision (Fixed)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torch.amp import autocast, GradScaler  # Fixed: use torch.amp instead of torch.cuda.amp
from torch.nn.utils import spectral_norm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Mixed Precision Scaler - Fixed API
scaler = GradScaler('cuda')
print("Mixed Precision (FP16) enabled!")

In [ ]:
# Load Dataset from Kaggle Input - DYNAMIC DETECTION
# Automatically detect dataset structure
KAGGLE_INPUT = '/kaggle/input'

# Find all image files recursively
image_paths = []
print(f"Scanning {KAGGLE_INPUT} for datasets...")

# List available datasets
for dataset_name in os.listdir(KAGGLE_INPUT):
    dataset_path = os.path.join(KAGGLE_INPUT, dataset_name)
    print(f"  Found dataset: {dataset_name}")
    
    # Recursively find all images
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                image_paths.append(os.path.join(root, file))

print(f"\nTotal images found: {len(image_paths)}")

if len(image_paths) == 0:
    print("\n⚠️ ERROR: No images found!")
    print("Make sure you have attached the 'islamic-patterns' dataset to this notebook.")
    print("Go to: Add Data -> Your Datasets -> islamic-patterns")
else:
    # Shuffle and Split
    random.seed(42)
    random.shuffle(image_paths)
    
    NUM_SAMPLES = len(image_paths)
    n_train = int(0.8 * NUM_SAMPLES)
    n_val = int(0.1 * NUM_SAMPLES)
    
    train_paths = image_paths[:n_train]
    val_paths = image_paths[n_train:n_train+n_val]
    test_paths = image_paths[n_train+n_val:]
    
    print(f"Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

In [ ]:
# Create Directories
DATA_DIR = '/kaggle/working/mughal_dataset'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
RESULTS_DIR = '/kaggle/working/results'

for d in [DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

for split in ['train', 'val', 'test']:
    for subdir in ['images', 'masks', 'edges']:
        os.makedirs(f'{DATA_DIR}/{split}/{subdir}', exist_ok=True)

print("Directories created!")

In [ ]:
# Mask Generation Functions
def generate_irregular_mask(height, width, min_strokes=1, max_strokes=4):
    """Generate irregular free-form masks to simulate damage."""
    mask = np.zeros((height, width), dtype=np.uint8)
    num_strokes = random.randint(min_strokes, max_strokes)
    
    for _ in range(num_strokes):
        start_x = random.randint(0, width - 1)
        start_y = random.randint(0, height - 1)
        
        for _ in range(random.randint(10, 30)):
            angle = random.uniform(0, 2 * np.pi)
            length = random.randint(10, 30)
            brush_w = random.randint(10, 40)
            
            end_x = int(start_x + length * np.cos(angle))
            end_y = int(start_y + length * np.sin(angle))
            end_x = np.clip(end_x, 0, width - 1)
            end_y = np.clip(end_y, 0, height - 1)
            
            cv2.line(mask, (start_x, start_y), (end_x, end_y), 255, brush_w)
            start_x, start_y = end_x, end_y
        
        if random.random() > 0.5:
            cx, cy = random.randint(0, width-1), random.randint(0, height-1)
            cv2.circle(mask, (cx, cy), random.randint(15, 50), 255, -1)
    
    return mask

def extract_canny_edges(image, low=50, high=150):
    """Extract edges using Canny."""
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image
    blurred = cv2.GaussianBlur(gray, (5, 5), 1.4)
    edges = cv2.Canny(blurred, low, high)
    return edges

print("Mask generation functions defined!")

In [ ]:
# Process Dataset
IMAGE_SIZE = 512  # Increased to match new dataset quality

def process_split(paths_list, split_name):
    success_count = 0
    for i, img_path in enumerate(tqdm(paths_list, desc=f'{split_name}')):
        try:
            img = Image.open(img_path).convert('RGB')
            img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
            img_np = np.array(img)
            
            mask = generate_irregular_mask(IMAGE_SIZE, IMAGE_SIZE)
            edges = extract_canny_edges(img_np)
            
            filename = f"{split_name}_{i:05d}.png"
            cv2.imwrite(f"{DATA_DIR}/{split_name}/images/{filename}", 
                       cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))
            cv2.imwrite(f"{DATA_DIR}/{split_name}/masks/{filename}", mask)
            cv2.imwrite(f"{DATA_DIR}/{split_name}/edges/{filename}", edges)
            success_count += 1
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            continue
    return success_count

print("Processing images...")
train_count = process_split(train_paths, 'train')
val_count = process_split(val_paths, 'val')
test_count = process_split(test_paths, 'test')

print(f"\nDataset prepared!")
print(f"Train: {train_count}, Val: {val_count}, Test: {test_count}")

In [ ]:
# Fast Fourier Convolution (FFC) Block - Core of LaMa (FP16 Compatible)
class FourierUnit(nn.Module):
    """Spectral transform for global receptive field - FP16 safe."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels * 2, out_channels * 2, 1, 1, 0)
        self.bn = nn.BatchNorm2d(out_channels * 2)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        batch = x.shape[0]
        
        # IMPORTANT: Cast to float32 for FFT (ComplexHalf is experimental and buggy)
        x_fp32 = x.float()
        
        # FFT in float32
        fft_x = torch.fft.rfft2(x_fp32, norm='ortho')
        fft_x = torch.cat([fft_x.real, fft_x.imag], dim=1)
        
        # Convolution in frequency domain (back to original dtype)
        fft_x = fft_x.to(x.dtype)
        out = self.conv(fft_x)
        out = self.bn(out)
        out = self.relu(out)
        
        # Split and inverse FFT (in float32)
        out_fp32 = out.float()
        c = out_fp32.shape[1] // 2
        out_complex = torch.complex(out_fp32[:, :c], out_fp32[:, c:])
        out_fp32 = torch.fft.irfft2(out_complex, s=x.shape[2:], norm='ortho')
        
        # Return in original dtype
        return out_fp32.to(x.dtype)

class FFCBlock(nn.Module):
    """Fast Fourier Convolution Block - combines local and global features."""
    def __init__(self, in_channels, out_channels, ratio_global=0.5):
        super().__init__()
        self.in_local = int(in_channels * (1 - ratio_global))
        self.in_global = in_channels - self.in_local
        self.out_local = int(out_channels * (1 - ratio_global))
        self.out_global = out_channels - self.out_local
        
        # Local to Local
        self.local_local = nn.Conv2d(self.in_local, self.out_local, 3, 1, 1) if self.in_local > 0 and self.out_local > 0 else None
        # Local to Global
        self.local_global = nn.Conv2d(self.in_local, self.out_global, 3, 1, 1) if self.in_local > 0 and self.out_global > 0 else None
        # Global to Local
        self.global_local = nn.Conv2d(self.in_global, self.out_local, 3, 1, 1) if self.in_global > 0 and self.out_local > 0 else None
        # Global to Global (Fourier)
        self.global_global = FourierUnit(self.in_global, self.out_global) if self.in_global > 0 and self.out_global > 0 else None
        
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x_local = x[:, :self.in_local] if self.in_local > 0 else None
        x_global = x[:, self.in_local:] if self.in_global > 0 else None
        
        out_local = 0
        out_global = 0
        
        if self.local_local is not None and x_local is not None:
            out_local = out_local + self.local_local(x_local)
        if self.global_local is not None and x_global is not None:
            out_local = out_local + self.global_local(x_global)
        if self.local_global is not None and x_local is not None:
            out_global = out_global + self.local_global(x_local)
        if self.global_global is not None and x_global is not None:
            out_global = out_global + self.global_global(x_global)
        
        if isinstance(out_local, int) and out_local == 0:
            out = out_global
        elif isinstance(out_global, int) and out_global == 0:
            out = out_local
        else:
            out = torch.cat([out_local, out_global], dim=1)
        
        out = self.bn(out)
        out = self.relu(out)
        return out

print("FFC Blocks defined (FP16 compatible)!")

In [ ]:
# LaMa Generator - Simplified Version
class LaMaGenerator(nn.Module):
    """LaMa-inspired generator with FFC blocks for global receptive field."""
    def __init__(self, in_channels=4, out_channels=3):
        super().__init__()
        
        # Initial convolution
        self.initial = nn.Sequential(
            nn.Conv2d(in_channels, 64, 7, 1, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        # Downsampling
        self.down1 = nn.Sequential(
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.down2 = nn.Sequential(
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        
        # FFC Residual Blocks (global receptive field)
        self.ffc_blocks = nn.Sequential(
            FFCBlock(256, 256),
            FFCBlock(256, 256),
            FFCBlock(256, 256),
            FFCBlock(256, 256),
            FFCBlock(256, 256),
            FFCBlock(256, 256),
        )
        
        # Upsampling
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        # Final convolution
        self.final = nn.Sequential(
            nn.Conv2d(64, out_channels, 7, 1, 3),
            nn.Tanh()
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    
    def forward(self, x, mask):
        inp = torch.cat([x, mask], dim=1)
        
        x = self.initial(inp)
        x = self.down1(x)
        x = self.down2(x)
        
        x = self.ffc_blocks(x)
        
        x = self.up1(x)
        x = self.up2(x)
        x = self.final(x)
        
        return x

print("LaMa Generator defined!")

In [ ]:
# Discriminator with Spectral Normalization
class Discriminator(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        # Apply spectral normalization to all conv layers for stable training
        self.model = nn.Sequential(
            spectral_norm(nn.Conv2d(in_channels, 64, 4, 2, 1)),
            nn.LeakyReLU(0.2, inplace=True),
            
            spectral_norm(nn.Conv2d(64, 128, 4, 2, 1)),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            spectral_norm(nn.Conv2d(128, 256, 4, 2, 1)),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            spectral_norm(nn.Conv2d(256, 512, 4, 2, 1)),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            spectral_norm(nn.Conv2d(512, 1, 4, 1, 1))
        )
    
    def forward(self, x):
        return self.model(x)

print("Discriminator with Spectral Normalization defined!")

In [ ]:
# Geometric Consistency Loss - Custom loss for geometric patterns (FP16 Safe)
def sobel_filter(x):
    """Apply Sobel filter to detect edges."""
    # Cast to float32 for stable computation
    x_fp32 = x.float()
    
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32, device=x.device).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32, device=x.device).view(1, 1, 3, 3)
    
    # Convert to grayscale if needed
    if x_fp32.shape[1] == 3:
        x_gray = 0.299 * x_fp32[:, 0:1] + 0.587 * x_fp32[:, 1:2] + 0.114 * x_fp32[:, 2:3]
    else:
        x_gray = x_fp32
    
    grad_x = F.conv2d(x_gray, sobel_x, padding=1)
    grad_y = F.conv2d(x_gray, sobel_y, padding=1)
    grad = torch.sqrt(grad_x ** 2 + grad_y ** 2 + 1e-6)
    
    return grad.to(x.dtype)

def geometric_consistency_loss(pred, target, mask):
    """
    Custom loss for geometric patterns.
    Penalizes broken lines and inconsistent edges.
    """
    # Cast to float32 for stable loss computation
    pred_fp32 = pred.float()
    target_fp32 = target.float()
    mask_fp32 = mask.float()
    
    # 1. Edge Continuity Loss
    pred_edges = sobel_filter(pred_fp32)
    target_edges = sobel_filter(target_fp32)
    edge_loss = F.l1_loss(pred_edges * mask_fp32, target_edges * mask_fp32)
    
    # 2. High-Frequency Loss (preserve fine details)
    laplacian = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32, device=pred.device).view(1, 1, 3, 3)
    
    pred_gray = 0.299 * pred_fp32[:, 0:1] + 0.587 * pred_fp32[:, 1:2] + 0.114 * pred_fp32[:, 2:3]
    target_gray = 0.299 * target_fp32[:, 0:1] + 0.587 * target_fp32[:, 1:2] + 0.114 * target_fp32[:, 2:3]
    
    pred_lap = F.conv2d(pred_gray, laplacian, padding=1)
    target_lap = F.conv2d(target_gray, laplacian, padding=1)
    hf_loss = F.l1_loss(pred_lap * mask_fp32, target_lap * mask_fp32)
    
    return edge_loss + 0.5 * hf_loss

print("Geometric Consistency Loss defined (FP16 safe)!")

In [ ]:
# Placeholder cell
pass

In [ ]:
# Dataset Class with Data Augmentation
class InpaintingDataset(Dataset):
    def __init__(self, data_dir, split='train', augment=True):
        self.image_dir = Path(data_dir) / split / 'images'
        self.mask_dir = Path(data_dir) / split / 'masks'
        self.image_files = sorted(list(self.image_dir.glob('*.png')))
        self.augment = augment and split == 'train'  # Only augment training data
        
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        mask_path = self.mask_dir / img_path.name
        
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')
        
        # Data Augmentation (rotation and flip)
        if self.augment:
            # Random 90-degree rotation (geometric patterns are rotationally symmetric)
            k = random.randint(0, 3)
            if k > 0:
                image = image.rotate(k * 90)
                mask = mask.rotate(k * 90)
            
            # Random horizontal flip
            if random.random() > 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            
            # Random vertical flip
            if random.random() > 0.5:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)
                mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
        
        image = self.transform(image)
        mask = transforms.ToTensor()(mask)
        
        # Create masked image
        masked_image = image * (1 - mask)
        
        return {
            'image': image,
            'masked_image': masked_image,
            'mask': mask
        }

# Create datasets with augmentation
train_dataset = InpaintingDataset(DATA_DIR, 'train', augment=True)
val_dataset = InpaintingDataset(DATA_DIR, 'val', augment=False)
test_dataset = InpaintingDataset(DATA_DIR, 'test', augment=False)

print(f"Train samples: {len(train_dataset)} (with augmentation: rotation + flip)")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Training Function
def train_lama_optimized(generator, disc, train_loader, val_loader, epochs=100, lr=0.0001):
    """LaMa training with Geometric Consistency Loss."""
    
    generator = generator.to(DEVICE)
    disc = disc.to(DEVICE)
    
    opt_gen = optim.Adam(generator.parameters(), lr=lr, betas=(0.0, 0.9))
    opt_disc = optim.Adam(disc.parameters(), lr=lr * 0.5, betas=(0.0, 0.9))

    scheduler_gen = optim.lr_scheduler.CosineAnnealingLR(opt_gen, T_max=epochs, eta_min=1e-6)
    scheduler_disc = optim.lr_scheduler.CosineAnnealingLR(opt_disc, T_max=epochs, eta_min=1e-7)
    
    l1_loss = nn.L1Loss()
    adv_loss = nn.BCEWithLogitsLoss()
    
    history = {'gen_loss': [], 'disc_loss': [], 'psnr': []}
    best_psnr = 0
    
    print("\n" + "="*60)
    print("TRAINING CONFIGURATION")
    print("="*60)
    print(f"Resolution: 512x512")
    print(f"Batch Size: {train_loader.batch_size}")
    print(f"Epochs: {epochs}")
    print(f"Learning Rate: {lr}")
    print(f"FP16 Mixed Precision: Enabled")
    print(f"Spectral Normalization: Enabled")
    print(f"Data Augmentation: Rotation + Flip")
    print(f"Geometric Loss Weight: 1.5x")
    print(f"Label Smoothing: 0.9/0.1")
    print(f"Gradient Clipping: 1.0")
    print("="*60 + "\n")
    
    for epoch in range(epochs):
        generator.train()
        disc.train()
        
        epoch_gen_loss = 0
        epoch_disc_loss = 0
        num_batches = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for batch in pbar:
            images = batch['image'].to(DEVICE)
            masked_images = batch['masked_image'].to(DEVICE)
            masks = batch['mask'].to(DEVICE)
            
            # Generator Training
            opt_gen.zero_grad()
            
            with autocast('cuda'):
                pred_images = generator(masked_images, masks)
                recon_loss = l1_loss(pred_images, images)
                geo_loss = geometric_consistency_loss(pred_images, images, masks)
                fake_pred = disc(pred_images)
                gen_adv_loss = adv_loss(fake_pred, torch.ones_like(fake_pred) * 0.9)
                total_gen_loss = recon_loss + 1.5 * geo_loss + 0.1 * gen_adv_loss
            
            scaler.scale(total_gen_loss).backward()
            scaler.unscale_(opt_gen)
            torch.nn.utils.clip_grad_norm_(generator.parameters(), 1.0)
            scaler.step(opt_gen)
            
            # Discriminator Training
            opt_disc.zero_grad()
            
            with autocast('cuda'):
                real_pred = disc(images)
                fake_pred = disc(pred_images.detach())
                disc_real_loss = adv_loss(real_pred, torch.ones_like(real_pred) * 0.9)
                disc_fake_loss = adv_loss(fake_pred, torch.ones_like(fake_pred) * 0.1)
                disc_loss = (disc_real_loss + disc_fake_loss) / 2
            
            scaler.scale(disc_loss).backward()
            scaler.unscale_(opt_disc)
            torch.nn.utils.clip_grad_norm_(disc.parameters(), 1.0)
            scaler.step(opt_disc)
            scaler.update()
            
            epoch_gen_loss += total_gen_loss.item()
            epoch_disc_loss += disc_loss.item()
            num_batches += 1
            
            # Update progress bar with metrics
            pbar.set_postfix({
                'gen': f'{total_gen_loss.item():.4f}',
                'geo': f'{geo_loss.item():.4f}',
                'disc': f'{disc_loss.item():.4f}'
            })
        
        avg_gen = epoch_gen_loss / num_batches
        avg_disc = epoch_disc_loss / num_batches
        
        # Validation PSNR
        generator.eval()
        psnr_total = 0
        num_valid = 0
        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(DEVICE)
                masked_images = batch['masked_image'].to(DEVICE)
                masks = batch['mask'].to(DEVICE)
                pred_images = generator(masked_images, masks)
                mse = torch.mean((pred_images.float() - images.float()) ** 2)
                if mse > 0:
                    psnr = 10 * torch.log10(4 / (mse + 1e-8))
                    psnr_total += psnr.item()
                    num_valid += 1
        
        avg_psnr = psnr_total / max(num_valid, 1)
        scheduler_gen.step()
        scheduler_disc.step()
        
        history['gen_loss'].append(avg_gen)
        history['disc_loss'].append(avg_disc)
        history['psnr'].append(avg_psnr)
        
        print(f"Epoch {epoch+1}: Gen={avg_gen:.4f}, Disc={avg_disc:.4f}, Val PSNR={avg_psnr:.2f} dB")
        
        # Save best model
        if avg_psnr > best_psnr:
            best_psnr = avg_psnr
            torch.save({
                'generator': generator.state_dict(),
                'disc': disc.state_dict(),
                'epoch': epoch + 1,
                'best_psnr': best_psnr,
            }, f'{CHECKPOINT_DIR}/best_model.pth')
            print(f" -> New best model saved! PSNR: {best_psnr:.2f} dB")
        
        # Save checkpoint every 20 epochs
        if (epoch + 1) % 20 == 0:
            torch.save({
                'generator': generator.state_dict(),
                'disc': disc.state_dict(),
                'epoch': epoch + 1,
                'psnr': avg_psnr,
            }, f'{CHECKPOINT_DIR}/checkpoint_epoch_{epoch+1}.pth')
            print(f" -> Checkpoint saved: epoch {epoch+1}")
    
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    print(f"Best PSNR: {best_psnr:.2f} dB")
    print(f"Baseline (OpenCV): 21.27 dB")
    print("="*60)
    
    return history

print("Training function defined!")

In [ ]:
# Training Setup
lama_gen = LaMaGenerator(in_channels=4, out_channels=3)
disc = Discriminator(in_channels=3)

BATCH_SIZE = 16
EPOCHS = 100

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Training: {len(train_dataset)} samples, {EPOCHS} epochs")
print(f"Resolution: 512x512, Batch: {BATCH_SIZE}")

history = train_lama_optimized(lama_gen, disc, train_loader, val_loader, epochs=EPOCHS)

In [ ]:
# Plot Training History
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['gen_loss'])
axes[0].set_title('Generator Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(history['disc_loss'])
axes[1].set_title('Discriminator Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')

axes[2].plot(history['psnr'])
axes[2].set_title('Validation PSNR')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('PSNR (dB)')
axes[2].axhline(y=21.27, color='r', linestyle='--', label='Baseline (OpenCV)')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_history.png', dpi=150)
plt.show()

print(f"\nBest PSNR achieved: {max(history['psnr']):.2f} dB")
print(f"Baseline (OpenCV): 21.27 dB")

In [ ]:
# Evaluation on Test Set
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Load best model
checkpoint = torch.load(f'{CHECKPOINT_DIR}/best_model.pth')
lama_gen.load_state_dict(checkpoint['generator'])
lama_gen.eval()

test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

psnr_scores = []
ssim_scores = []
mae_scores = []

print("Evaluating on test set...")

with torch.no_grad():
    for batch in tqdm(test_loader):
        images = batch['image'].to(DEVICE)
        masked_images = batch['masked_image'].to(DEVICE)
        masks = batch['mask'].to(DEVICE)
        
        pred_images = lama_gen(masked_images, masks)
        
        # Convert to numpy
        pred_np = ((pred_images[0].cpu().numpy() + 1) / 2 * 255).astype(np.uint8).transpose(1, 2, 0)
        target_np = ((images[0].cpu().numpy() + 1) / 2 * 255).astype(np.uint8).transpose(1, 2, 0)
        
        # Calculate metrics
        psnr_val = psnr(target_np, pred_np)
        ssim_val = ssim(target_np, pred_np, multichannel=True, channel_axis=2)
        mae_val = np.mean(np.abs(target_np.astype(float) - pred_np.astype(float)))
        
        psnr_scores.append(psnr_val)
        ssim_scores.append(ssim_val)
        mae_scores.append(mae_val)

print("\n" + "="*50)
print("EVALUATION RESULTS (LaMa + Geometric Loss)")
print("="*50)
print(f"PSNR: {np.mean(psnr_scores):.4f} ± {np.std(psnr_scores):.4f}")
print(f"SSIM: {np.mean(ssim_scores):.4f} ± {np.std(ssim_scores):.4f}")
print(f"MAE: {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print("="*50)
print("\nBASELINE (OpenCV Navier-Stokes)")
print("="*50)
print(f"PSNR: 21.27")
print(f"SSIM: 0.842")
print(f"MAE: 8.57")
print("="*50)

improvement = np.mean(psnr_scores) - 21.27
print(f"\nImprovement: {improvement:+.2f} dB PSNR")

In [ ]:
# Visualize Results
def visualize_results(model, dataset, num_samples=5):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    indices = random.sample(range(len(dataset)), num_samples)
    
    for i, idx in enumerate(indices):
        sample = dataset[idx]
        image = sample['image'].unsqueeze(0).to(DEVICE)
        masked_image = sample['masked_image'].unsqueeze(0).to(DEVICE)
        mask = sample['mask'].unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            pred = model(masked_image, mask)
        
        # Convert to display format
        def to_display(t):
            return ((t[0].cpu().numpy() + 1) / 2).transpose(1, 2, 0).clip(0, 1)
        
        axes[i, 0].imshow(to_display(image))
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(to_display(masked_image))
        axes[i, 1].set_title('Damaged')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(to_display(pred))
        axes[i, 2].set_title('LaMa Restored')
        axes[i, 2].axis('off')
        
        axes[i, 3].imshow(mask[0, 0].cpu().numpy(), cmap='gray')
        axes[i, 3].set_title('Mask')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/lama_results.png', dpi=150)
    plt.show()

visualize_results(lama_gen, test_dataset)

In [ ]:
# Final Summary
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Model: LaMa (Fast Fourier Convolutions)")
print(f"Custom Loss: Geometric Consistency Loss")
print(f"Epochs: {EPOCHS}")
print(f"Dataset: {len(train_dataset) + len(val_dataset) + len(test_dataset)} images")
print("\nRESULTS:")
print(f"  LaMa PSNR: {np.mean(psnr_scores):.2f} dB")
print(f"  Baseline PSNR: 21.27 dB")
print(f"  Improvement: {np.mean(psnr_scores) - 21.27:+.2f} dB")
print("="*60)